In [95]:
# %% [markdown]
# Helper functions to read NetCDF, select time index, and compute errors

# %%
import netCDF4 as nc
import numpy as np
from pathlib import Path


def open_dataset(path_str):
    """
    Open a NetCDF dataset from the given path.
    """
    path = Path(path_str).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")
    return nc.Dataset(path, mode="r")


def normalize_time_index(time_index, ntime):
    """
    Normalize a user-provided time index.

    - Converts to int
    - Allows -1 to mean the last time index
    - Checks bounds
    """
    idx = int(time_index)

    if idx == -1:
        idx = ntime - 1

    if idx < 0 or idx >= ntime:
        raise IndexError(
            f"time index {idx} out of bounds [0, {ntime - 1}] "
            "(note: -1 means the last index before normalization)"
        )

    return idx


def compute_l1_l2_error(state_t, state_0):
    """
    Compute L1 and L2 errors of state_t relative to state_0.

    Definitions used:
        L1 = mean(|state_t - state_0|)
        L2 = sqrt( mean( (state_t - state_0)^2 ) )

    Returns
    -------
    l1 : float
    l2 : float
    """
    diff = state_t - state_0
    diff_flat = diff.ravel().astype(np.float64)

    l1 = np.mean(np.abs(diff_flat))
    l2 = np.sqrt(np.mean(diff_flat**2))

    return l1, l2


In [96]:
# %% [markdown]
# === USER INPUT SECTION ===
# Edit the variables below to point to your files, desired variable, and time.

file_1_suffix = "_N_10_f_10_QG_hydro_bgVel_0_with_imbalance.nc"
file_2_suffix = "_N_10_f_10_QG_on_bgVel_0_with_imbalance.nc"
# %%
# Paths to the two NetCDF analysis/checkpoint files
file1_path = ("/home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_10_40" + file_1_suffix)   # <-- EDIT
file2_path = ("/home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_10_40" + file_2_suffix)  # <-- EDIT

# Variable name to analyze (must exist in both files)
# e.g. "rho", "u", "Y", "rhoY", "p2_nodes", ...
var_name = "p2_nodes"  # <-- EDIT

# Target time:
#   If use_time_index = False: physical time value (same units as 'time' var in file)
#   If use_time_index = True: integer time index
target_time = 50 # <-- EDIT (e.g. t = 10.0)
ensemble_time = False
use_time_index = False  # Set to True if you want to specify target_time as an index

print("File 1:", file1_path)
print("File 2:", file2_path)
print("Variable:", var_name)
print("Target time:", target_time, "(time index mode:" , use_time_index, ")")


File 1: /home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_10_40_N_10_f_10_QG_hydro_bgVel_0_with_imbalance.nc
File 2: /home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_10_40_N_10_f_10_QG_on_bgVel_0_with_imbalance.nc
Variable: p2_nodes
Target time: 50 (time index mode: False )


In [97]:
# %% [markdown]
# Compute L1 and L2 errors from initial state (time index = 0) for each file.

# %%
# Open datasets
ds1 = open_dataset(file1_path)
ds2 = open_dataset(file2_path)

print("Opened datasets successfully.")


def analyze_file(ds, label, var_name, time_index_raw):
    """
    For a given dataset, compute L1/L2 error of variable `var_name`
    at the given time index relative to time index 0.

    Time dimension is assumed to be the FIRST axis of the variable:
        shape = (x, n, m, k)
    where x is the time index.
    """
    if var_name not in ds.variables:
        raise KeyError(f"Variable '{var_name}' not found in dataset.")

    var = ds.variables[var_name]
    if len(var.shape) == 0:
        raise ValueError(f"Variable '{var_name}' is scalar, no time dimension.")

    # Total number of time indices
    ntime = var.shape[0]

    # Normalize the time index (handle -1, bounds, etc.)
    time_index = normalize_time_index(time_index_raw, ntime)

    # Initial state at time index 0
    state_0 = np.array(var[0, ...])

    # State at requested time index
    state_t = np.array(var[time_index, ...])

    # Compute error norms
    l1, l2 = compute_l1_l2_error(state_t, state_0)

    print(f"=== {label} ===")
    print(f"Variable            : {var_name}")
    print(f"Variable Shape      : {var.shape}")
    print(f"Number of time steps: {ntime}")
    print(f"Initial time index  : 0")
    print(f"Requested time index: user input = {time_index_raw} -> used = {time_index}")
    print(f"L1 error (mean |Δ|) = {l1:e}")
    print(f"L2 error (RMS)      = {l2:e}")
    print()


# Interpret target_time from Cell 2 as a time index
time_index_input = int(target_time)

# Analyze first file
analyze_file(ds1, "FILE 1", var_name, time_index_input)

# Analyze second file
analyze_file(ds2, "FILE 2", var_name, time_index_input)


Opened datasets successfully.
=== FILE 1 ===
Variable            : p2_nodes
Variable Shape      : (203, 41, 11, 41)
Number of time steps: 203
Initial time index  : 0
Requested time index: user input = 50 -> used = 50
L1 error (mean |Δ|) = 1.209259e-05
L2 error (RMS)      = 2.621622e-05

=== FILE 2 ===
Variable            : p2_nodes
Variable Shape      : (203, 41, 11, 41)
Number of time steps: 203
Initial time index  : 0
Requested time index: user input = 50 -> used = 50
L1 error (mean |Δ|) = 1.091117e-05
L2 error (RMS)      = 2.757662e-05



In [98]:
# %%
ds1.close()
ds2.close()
print("Datasets closed.")


Datasets closed.
